# 04 - Validate Standard Catalog Outputs

        Validate that the Silver and Gold tables exist, have rows, preserve the PO-line grain, and produce the expected outcomes for the deterministic demo cases.

In [ ]:
# Edit these values for your AIDP workspace.
SOURCE_CATALOG = "aidp_sc_demo_source"
SOURCE_SCHEMA = "aidp_sc_demo"  # Use AIDP_SC_DEMO if your workspace exposes uppercase schema names.

TARGET_CATALOG = "aidp_sc_demo_standard"
SILVER_SCHEMA = "demo_supply_chain_silver"
GOLD_SCHEMA = "demo_supply_chain_gold"

# AIDP standard catalogs use managed Delta tables. Keep this as "delta" unless your tenancy requires a different table format.
TABLE_FORMAT = "delta"

In [ ]:
import re
from pyspark.sql import functions as F

IDENTIFIER_PATTERN = re.compile(r"^[A-Za-z_][A-Za-z0-9_]*$")


def qident(value: str) -> str:
    """Quote and validate a catalog, schema, or table identifier."""
    if not IDENTIFIER_PATTERN.fullmatch(value):
        raise ValueError(f"Unsupported identifier: {value!r}")
    return f"`{value}`"


def qname(*parts: str) -> str:
    return ".".join(qident(part) for part in parts)


def show_small(df, n: int = 20) -> None:
    """Display a small result in notebook UI, falling back to show()."""
    try:
        display(df.limit(n))
    except NameError:
        df.show(n, truncate=False)


SOURCE_TABLES = {
    "suppliers": "aidp_sc_suppliers",
    "supplier_sites": "aidp_sc_supplier_sites",
    "item_categories": "aidp_sc_item_categories",
    "items": "aidp_sc_items",
    "po_headers": "aidp_sc_po_headers",
    "po_lines": "aidp_sc_po_lines",
    "blanket_prices": "aidp_sc_blanket_prices",
    "receipts": "aidp_sc_receipts",
    "invoice_lines": "aidp_sc_invoice_lines",
}


def source_table(key: str):
    return spark.table(qname(SOURCE_CATALOG, SOURCE_SCHEMA, SOURCE_TABLES[key]))


def normalize_table_name(table: str) -> str:
    """AIDP standard catalog table names are easiest to resolve consistently in lowercase."""
    normalized = table.lower()
    if not IDENTIFIER_PATTERN.fullmatch(normalized):
        raise ValueError(f"Unsupported table identifier: {table!r}")
    return normalized


def target_table(schema: str, table: str) -> str:
    return qname(TARGET_CATALOG, schema, normalize_table_name(table))


def ensure_schema(schema: str) -> None:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qname(TARGET_CATALOG, schema)}")


def use_target_schema(schema: str) -> None:
    """Set the active catalog/schema before standard-catalog table access.

    AIDP notebooks can be stricter about three-part table resolution than local Spark.
    Setting the active catalog and schema before reads/writes avoids losing the target
    catalog during Delta table lookup.
    """
    spark.sql(f"USE CATALOG {qident(TARGET_CATALOG)}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {qident(schema)}")
    spark.sql(f"USE SCHEMA {qident(schema)}")


def read_managed_table(schema: str, table: str):
    table_name = normalize_table_name(table)
    use_target_schema(schema)
    return spark.table(qident(table_name))

## Confirm target tables and row counts

In [ ]:
target_tables = [
    (SILVER_SCHEMA, "SUPPLY_PO_LINE_COMPARISON"),
    (SILVER_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_FEATURES"),
    (GOLD_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_OUTPUT"),
]

row_counts = []
for schema_name, table_name in target_tables:
    full_name = target_table(schema_name, table_name)
    count_value = read_managed_table(schema_name, table_name).count()
    row_counts.append((schema_name, table_name, count_value))
    if count_value == 0:
        raise RuntimeError(f"{full_name} has zero rows")

show_small(spark.createDataFrame(row_counts, ["schema_name", "table_name", "row_count"]))

## Validate PO-line grain

In [ ]:
comparison = read_managed_table(SILVER_SCHEMA, "SUPPLY_PO_LINE_COMPARISON")
gold = read_managed_table(GOLD_SCHEMA, "SUPPLY_PO_PRICE_REVIEW_OUTPUT")

def assert_unique_po_line(df, label: str) -> None:
    duplicates = (
        df.groupBy("po_number", "line_number")
        .count()
        .where(F.col("count") > 1)
    )
    duplicate_count = duplicates.count()
    if duplicate_count:
        show_small(duplicates, 20)
        raise RuntimeError(f"{label} contains duplicate PO number / line number rows")

assert_unique_po_line(comparison, "SUPPLY_PO_LINE_COMPARISON")
assert_unique_po_line(gold, "SUPPLY_PO_PRICE_REVIEW_OUTPUT")
print("PO-line grain is unique in Silver and Gold outputs.")

## Validate deterministic demo outcomes

In [ ]:
expected_cases = [
    ("NORMAL", "PO-DEMO-NORMAL", 1, "NORMAL", "RECEIVED", "MATCHED", "MONITOR"),
    ("PRICE", "PO-DEMO-PRICE", 1, "HIGH", "RECEIVED", "MATCHED", "BLANKET_PRICE_REVIEW"),
    ("REJECT", "PO-DEMO-REJECT", 1, "NORMAL", "REJECTED", "MATCHED", "RECEIVING_REVIEW"),
    ("INVOICE", "PO-DEMO-INVOICE", 1, "NORMAL", "RECEIVED", "PRICE_MISMATCH", "INVOICE_MATCH_REVIEW"),
    ("COMBINED", "PO-DEMO-COMBINED", 1, "HIGH", "REJECTED", "PRICE_MISMATCH", "COMBINED_PROCUREMENT_REVIEW"),
    ("NO-RECEIPT", "PO-DEMO-NO-RECEIPT", 1, "NORMAL", "NOT_FOUND", "MATCHED", "GATHER_EVIDENCE"),
    ("NO-INVOICE", "PO-DEMO-NO-INVOICE", 1, "NORMAL", "RECEIVED", "NOT_FOUND", "GATHER_EVIDENCE"),
    ("UOM", "PO-DEMO-UOM", 1, "NOT_COMPARABLE", "RECEIVED", "MATCHED", "GATHER_EVIDENCE"),
    ("CURRENCY", "PO-DEMO-CURRENCY", 1, "NOT_COMPARABLE", "RECEIVED", "MATCHED", "GATHER_EVIDENCE"),
    ("PARTIAL", "PO-DEMO-PARTIAL", 1, "NORMAL", "PARTIAL", "MATCHED", "RECEIVING_REVIEW"),
]

expected_df = spark.createDataFrame(
    expected_cases,
    ["case_id", "po_number", "line_number", "expected_price_signal", "expected_receipt_signal", "expected_invoice_signal", "expected_review_outcome"],
)

actual_df = gold.select(
    "po_number",
    "line_number",
    F.col("price_signal").alias("actual_price_signal"),
    F.col("receipt_signal").alias("actual_receipt_signal"),
    F.col("invoice_signal").alias("actual_invoice_signal"),
    F.col("review_outcome").alias("actual_review_outcome"),
)

validation_df = (
    expected_df
    .join(actual_df, ["po_number", "line_number"], "left")
    .withColumn(
        "validation_status",
        F.when(F.col("actual_review_outcome").isNull(), F.lit("MISSING"))
        .when(
            (F.col("expected_price_signal") == F.col("actual_price_signal"))
            & (F.col("expected_receipt_signal") == F.col("actual_receipt_signal"))
            & (F.col("expected_invoice_signal") == F.col("actual_invoice_signal"))
            & (F.col("expected_review_outcome") == F.col("actual_review_outcome")),
            F.lit("PASS"),
        )
        .otherwise(F.lit("FAIL")),
    )
)

show_small(validation_df.orderBy("case_id"), 20)

failed = validation_df.where(F.col("validation_status") != "PASS").count()
if failed:
    raise RuntimeError(f"{failed} deterministic demo case validations failed")

print("All deterministic demo cases passed.")

## Result

        Validation is complete when row counts are nonzero, PO-line grain is unique, and all deterministic demo cases pass.